## Text Cleaning

In [1]:
# import libraries
import pandas as pd
import numpy as np
import nltk
import re

In [2]:
# load dataset
df = pd.read_csv('data/headlines.csv')

print(df.head())

                                           headline       category
0   Market sees strong growth during national event       Business
1     Actor stars in new film during national event  Entertainment
2  Award winners announced following recent changes  Entertainment
3               League match ends in draw this week         Sports
4      Doctors recommend screening as experts react         Health


In [3]:
# check missing values
print(df.isnull().sum())

headline    0
category    0
dtype: int64


In [4]:
# lowercase 
df['headline'] = df['headline'].str.lower()
print(df.head())

                                           headline       category
0   market sees strong growth during national event       Business
1     actor stars in new film during national event  Entertainment
2  award winners announced following recent changes  Entertainment
3               league match ends in draw this week         Sports
4      doctors recommend screening as experts react         Health


In [5]:
# remove punctuation, numbers, and special characters
df['headline'] = df['headline'].apply(lambda headline: re.sub(r'[^A-Za-z]', ' ', headline))
print(df.head())

                                           headline       category
0   market sees strong growth during national event       Business
1     actor stars in new film during national event  Entertainment
2  award winners announced following recent changes  Entertainment
3               league match ends in draw this week         Sports
4      doctors recommend screening as experts react         Health


In [6]:
# tokenization
from nltk.tokenize import word_tokenize

df['headline'] = df['headline'].apply(word_tokenize)
print(df.head())

                                            headline       category
0  [market, sees, strong, growth, during, nationa...       Business
1  [actor, stars, in, new, film, during, national...  Entertainment
2  [award, winners, announced, following, recent,...  Entertainment
3        [league, match, ends, in, draw, this, week]         Sports
4  [doctors, recommend, screening, as, experts, r...         Health


In [7]:
# remove stop words
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
df['headline'] = df['headline'].apply(lambda tokens: [w for w in tokens if w not in stop_words])
print(df.head())

                                            headline       category
0    [market, sees, strong, growth, national, event]       Business
1         [actor, stars, new, film, national, event]  Entertainment
2  [award, winners, announced, following, recent,...  Entertainment
3                  [league, match, ends, draw, week]         Sports
4    [doctors, recommend, screening, experts, react]         Health


In [8]:
# lemmatization
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

def get_wordnet_pos(pos_tag):
    first_letter = pos_tag[0]

    if first_letter == 'J':
        return wordnet.ADJ
    elif first_letter == 'V':
        return wordnet.VERB
    elif first_letter == 'R':
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
lemmatizer = WordNetLemmatizer()

df['headline'] = df['headline'].apply(lambda tokens: [ 
    lemmatizer.lemmatize(word, get_wordnet_pos(pos_tag)) 
    for (word, pos_tag) in nltk.pos_tag(tokens)
    ])
print(df.head())

                                            headline       category
0     [market, see, strong, growth, national, event]       Business
1          [actor, star, new, film, national, event]  Entertainment
2  [award, winner, announce, follow, recent, change]  Entertainment
3                   [league, match, end, draw, week]         Sports
4         [doctor, recommend, screen, expert, react]         Health


In [9]:
# join words
df['headline'] = df['headline'].apply(lambda tokens: ' '.join(tokens))
print(df.head())

                                     headline       category
0     market see strong growth national event       Business
1          actor star new film national event  Entertainment
2  award winner announce follow recent change  Entertainment
3                  league match end draw week         Sports
4        doctor recommend screen expert react         Health


In [10]:
# X and y
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X = df['headline']
y = le.fit_transform(df['category'])

In [11]:
# train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)